# 第7节：视频编码原理（DCT + 量化）

本 Notebook 包含三个实验，帮助你理解视频编码的核心原理。

**实验内容：**
1. 手写 8x8 DCT
2. 量化与反量化
3. 观察压缩前后的图像变化

## 环境准备

确保已安装 NumPy 和 Matplotlib。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print(f"NumPy 版本: {np.__version__}")
print(f"Matplotlib 版本: {plt.matplotlib.__version__}")

## 实验1：手写 8x8 DCT

**目标**：理解 DCT 变换的原理

In [ ]:
def dct_1d(signal):
    """一维 DCT 变换"""
    N = len(signal)
    result = np.zeros(N)
    
    for k in range(N):
        sum_val = 0
        for n in range(N):
            sum_val += signal[n] * np.cos(np.pi * (2 * n + 1) * k / (2 * N))
        
        if k == 0:
            result[k] = sum_val * np.sqrt(1 / N)
        else:
            result[k] = sum_val * np.sqrt(2 / N)
    
    return result

def dct_2d(block):
    """二维 DCT 变换（先对行做 DCT，再对列做 DCT）"""
    rows, cols = block.shape
    
    # 对每一行做 DCT
    temp = np.zeros_like(block, dtype=float)
    for i in range(rows):
        temp[i, :] = dct_1d(block[i, :].astype(float))
    
    # 对每一列做 DCT
    result = np.zeros_like(block, dtype=float)
    for j in range(cols):
        result[:, j] = dct_1d(temp[:, j])
    
    return result

def idct_1d(signal):
    """一维逆 DCT 变换"""
    N = len(signal)
    result = np.zeros(N)
    
    for n in range(N):
        sum_val = 0
        for k in range(N):
            if k == 0:
                sum_val += signal[k] * np.sqrt(1 / N)
            else:
                sum_val += signal[k] * np.sqrt(2 / N) * np.cos(np.pi * (2 * n + 1) * k / (2 * N))
        
        result[n] = sum_val
    
    return result

def idct_2d(dct_coeffs):
    """二维逆 DCT 变换"""
    rows, cols = dct_coeffs.shape
    
    # 对每一列做逆 DCT
    temp = np.zeros_like(dct_coeffs, dtype=float)
    for j in range(cols):
        temp[:, j] = idct_1d(dct_coeffs[:, j])
    
    # 对每一行做逆 DCT
    result = np.zeros_like(dct_coeffs, dtype=float)
    for i in range(rows):
        result[i, :] = idct_1d(temp[i, :])
    
    return result

# 测试 DCT
print("=" * 50)
print("DCT 变换实验")
print("=" * 50)

# 创建测试块1：平滑梯度
block_smooth = np.array([
    [120, 122, 125, 128, 130, 132, 135, 138],
    [121, 123, 126, 129, 131, 133, 136, 139],
    [122, 124, 127, 130, 132, 134, 137, 140],
    [123, 125, 128, 131, 133, 135, 138, 141],
    [124, 126, 129, 132, 134, 136, 139, 142],
    [125, 127, 130, 133, 135, 137, 140, 143],
    [126, 128, 131, 134, 136, 138, 141, 144],
    [127, 129, 132, 135, 137, 139, 142, 145]
], dtype=np.uint8)

# 创建测试块2：垂直边缘
block_edge = np.array([
    [100, 100, 100, 100, 200, 200, 200, 200],
    [100, 100, 100, 100, 200, 200, 200, 200],
    [100, 100, 100, 100, 200, 200, 200, 200],
    [100, 100, 100, 100, 200, 200, 200, 200],
    [100, 100, 100, 100, 200, 200, 200, 200],
    [100, 100, 100, 100, 200, 200, 200, 200],
    [100, 100, 100, 100, 200, 200, 200, 200],
    [100, 100, 100, 100, 200, 200, 200, 200]
], dtype=np.uint8)

# 测试平滑块
print("\n--- 测试块1：平滑梯度 ---")
print(f"原始像素块 (8x8):")
print(block_smooth)

dct_smooth = dct_2d(block_smooth)
print(f"\nDCT 系数 (8x8):")
print(np.round(dct_smooth, 1))

# 测试边缘块
print("\n--- 测试块2：垂直边缘 ---")
print(f"原始像素块 (8x8):")
print(block_edge)

dct_edge = dct_2d(block_edge)
print(f"\nDCT 系数 (8x8):")
print(np.round(dct_edge, 1))

# 验证 DCT/IDCT 可逆性
print("\n--- DCT/IDCT 可逆性验证 ---")
reconstructed = idct_2d(dct_smooth)
if np.allclose(block_smooth.astype(float), reconstructed, atol=1e-10):
    print("✓ DCT/IDCT 可逆性验证通过")
else:
    print("✗ DCT/IDCT 可逆性验证失败")

## 实验2：量化与反量化

**目标**：理解量化的作用和对画质的影响

In [ ]:
def quantize(dct_coeffs, qp=23):
    """量化 DCT 系数（教学简化版本）"""
    # JPEG 标准亮度量化矩阵
    q_matrix = np.array([
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99]
    ])
    
    # 教学简化：QP 每增加 6，量化步长翻倍
    # 实际 H.264 使用 Qstep = 2^(QP/6)
    q_step = q_matrix * (2 ** (qp / 6))
    
    # 量化
    quantized = np.round(dct_coeffs / q_step).astype(int)
    
    return quantized, q_step

def dequantize(quantized, q_step):
    """反量化"""
    return quantized * q_step

# 测试量化
print("=" * 50)
print("量化实验")
print("=" * 50)

# 使用边缘块进行测试
dct_coeffs = dct_edge

# 使用不同的 QP 进行量化
for qp in [0, 12, 24, 36, 48]:
    quantized, q_step = quantize(dct_coeffs, qp)
    dequantized = dequantize(quantized, q_step)
    
    print(f"\nQP = {qp}:")
    print(f"  量化后非零系数: {np.count_nonzero(quantized)}")
    print(f"  量化步长范围: {q_step.min():.1f} - {q_step.max():.1f}")

## 实验3：观察压缩前后的图像变化

**目标**：理解压缩失真的原因

In [ ]:
# 可视化对比
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# 原始图像
axes[0, 0].imshow(block_edge, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

# 不同 QP 的压缩结果（4个QP，填满2x3子图）
qp_values = [0, 12, 24, 36]
positions = [(0, 1), (0, 2), (1, 1), (1, 2)]  # 预定义位置

for idx, qp in enumerate(qp_values):
    row, col = positions[idx]
    
    quantized, q_step = quantize(dct_coeffs, qp)
    dequantized = dequantize(quantized, q_step)
    reconstructed = idct_2d(dequantized)
    
    axes[row, col].imshow(np.clip(reconstructed, 0, 255).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
    axes[row, col].set_title(f'QP={qp}')
    axes[row, col].axis('off')

# 隐藏空白子图
axes[1, 0].axis('off')

plt.tight_layout()
plt.savefig('dct_comparison.png', dpi=150)
plt.show()

print("对比图已保存到 dct_comparison.png")

## 总结

通过本实验，你应该掌握了：

1. **DCT 变换**
   - 将空间域转换为频率域
   - 低频系数代表主要能量，高频系数代表细节
   - 平滑块只有 DC 系数非零，边缘块有多个非零系数

2. **量化**
   - 将 DCT 系数除以量化步长并取整
   - QP 越大，量化越狠，压缩率越高，画质越差
   - QP 每增加 6，量化步长翻倍

3. **压缩失真**
   - 量化丢弃高频细节，导致图像模糊
   - QP 是控制画质和码率的核心参数